# INF 791 - Tópicos Especiais II – Redes Complexas
## Trabalho Prático - Intermediário

Este notebook foi desenvolvido para resolver passo a passo as questões do trabalho prático.


# 1. Coleta de Dados

## (a) Escolha do Sistema e Construção do Coletor do Grafo

### Base de dados escolhida  
A base de dados escolhida para esta análise é a Social circles: Facebook (*Facebook Ego Networks*), obtida através da coleção pública *SNAP (Stanford Large Network Dataset Collection)* e coletada originalmente em 2012 pelos pesquisadores Jure Leskovec e Julian McAuley. 

É importante destacar que **não estamos consultando a base total do Facebook em tempo real** (a qual é estritamente privada por segredo comercial e segurança). Em vez disso, estamos utilizando um dataset histórico e estático composto pela fusão de **10 redes egocêntricas (ego-networks)** distintas. 

Uma ego-network é a rede formada por um usuário focal (o "Ego"), todos os seus contatos diretos (amigos), e todas as conexões existentes entre esses próprios contatos. Esses dados foram obtidos de voluntários reais que aceitaram mapear seus círculos de amizades de forma totalmente anonimizada para fins acadêmicos.

Dessa forma, nosso grafo com 4.039 nós e 88.234 arestas representa uma amostragem estruturada localmente densa do Facebook daquela época (cerca de 0,0004% da base total). A grande vantagem desse recorte é que ele preserva a riqueza de comunidades fechadas e círculos sociais originais do comportamento humano real, o que seria perdido em uma amostragem puramente aleatória.
 
A proposta em relação a este grafo é comprovar empiricamente propriedades clássicas e leis universais das redes sociais reais estudadas em sala de aula. Partindo da rede formada por esse dataset, serão avaliadas as seguintes métricas de redes complexas:

### Grau Médio e Distribuição de Grau  
Esta métrica quantifica a popularidade individual de cada nó, determinando exatamente a quantidade de amigos de um usuário. Será utilizada para expor a desigualdade de popularidade na rede. 

O objetivo com essa métrica é verificar se o grafo apresenta um comportamento Livre de Escala (*Scale-Free*), caracterizado por uma minoria de usuários com um número gigantesco de ligações.

### Coeficiente de Clusterização  
Através do coeficiente de clusterização vamos medir matematicamente a probabilidade de que dois amigos de um determinado usuário também sejam amigos entre si. 

Utilizaremos esse cálculo na análise para comprovar o princípio do "Fechamento Triádico", atestando se a rede do Facebook possui uma tendência à formação de comunidades isoladas, panelinhas e bolhas sociais densas.

### Distância Média  
A distância média será calculada para encontrar o número mínimo de conexões necessárias para que uma informação vá de um usuário qualquer até outro dentro do grafo. 

Essa métrica servirá para comprovar de forma empírica a teoria dos "Seis Graus de Separação" e demonstrar o fenômeno do Mundo Pequeno (*Small-World*).

### Sobreposição de Vizinhança (Overlap)  
O *overlap* medirá a quantidade percentual de amigos que duas pessoas conectadas possuem em comum. Nas análises, essa métrica será o pilar para diferenciar e validar a teoria sociológica dos laços. Conseguiremos identificar quais arestas são "Laços Fortes" e quais são "Laços Fracos".

### Componentes Conexos  
Por fim, o cálculo de componentes conexos identificará se existem ilhas de usuários completamente isoladas do resto do grafo e qual o tamanho exato de cada um desses blocos. A análise partirá dessa contagem para confirmar a tendência das redes de conectarem a vasta maioria de seus participantes em um único "Componente Gigante" contínuo.

### Obtenção dos Dados (API vs Base Pública):  
Historicamente, dados de listas de amigos do Facebook eram extraídos via *Facebook Graph API*. No entanto, devido às grandes restrições de privacidade das plataformas atualmente, essa coleta tornou-se impossível.

Dessa forma o presente trabalho irá explorar a base pública do projeto acadêmico de Stanford (SNAP), que fornece esses dados de conectividade de maneira totalmente anonimizada e em larga escala.
* Fonte dos dados: http://snap.stanford.edu/data/ego-Facebook.html

### Construção do Coletor e Chamadas Utilizadas:  
Para baixar e processar os dados de forma automatizada, o código realiza as seguintes operações:

1. **Verificação de Arquivo (`os.path.exists`)**: Verifica se o arquivo final descompactado (`facebook_combinado.txt`) já existe localmente para evitar downloads repetidos.
2. **Download dos Dados (`requests.get`)**: Caso não exista, faz o download do dataset compactado (`facebook_combinado.txt.gz`) a partir do repositório público do SNAP.
3. **Extração e Descompactação (`gzip.open` e `shutil.copyfileobj`)**: Abre o arquivo compactado e extrai o seu conteúdo para um arquivo de texto plano.
4. **Construção do Grafo (`nx.read_edgelist`)**: Lê a lista de conexões do arquivo de texto e modela a rede como um grafo não-direcionado (`nx.Graph()`) na variável `grafo`, definindo os nós como números inteiros (`nodetype=int`) para otimizar o desempenho.


In [ ]:
import networkx as nx
import requests
import gzip
import shutil
import os

# Configurações de URL e arquivos locais
url_dados = "https://snap.stanford.edu/data/facebook_combined.txt.gz"
arquivo_gz = "facebook_combinado.txt.gz"
arquivo_txt = "facebook_combinado.txt"

# 1. Coleta / Download dos dados brutos via HTTP
if not os.path.exists(arquivo_txt):
    print("Iniciando coleta a partir do repositório público (SNAP)...")
    resposta = requests.get(url_dados, stream=True)
    if resposta.status_code == 200:
        with open(arquivo_gz, 'wb') as arq_saida:
            shutil.copyfileobj(resposta.raw, arq_saida)
        print("Download dos dados concluído!")
        
        # 2. Descompactação
        print("Extraindo o dataset...")
        with gzip.open(arquivo_gz, 'rb') as arq_entrada:
            with open(arquivo_txt, 'wb') as arq_saida:
                shutil.copyfileobj(arq_entrada, arq_saida)
        print("Extração concluída!")
    else:
        print(f"Falha ao baixar o arquivo. Código de status: {resposta.status_code}")
else:
    print("O dataset já foi coletado e está presente no diretório local.")

# 3. Processamento: Construção do Grafo Não-Direcionado
print("\nMontando a rede complexa na memória...")
grafo = nx.read_edgelist(arquivo_txt, create_using=nx.Graph(), nodetype=int)

print(f"Grafo construído com sucesso!")
print(f"-> Total de Nós (Usuários anonimizados): {grafo.number_of_nodes()}")
print(f"-> Total de Arestas (Relações de amizade): {grafo.number_of_edges()}")


## (b) Identificação e Discussão do Grafo Estudado

### Grafo estudado
O grafo selecionado para este estudo consiste na união de 10 redes egocêntricas (ou *ego-networks*) extraídas de perfis reais do Facebook, contendo 4.039 usuários (nós) e 88.234 relações de amizade mútuas (arestas). Trata-se de uma rede social não-direcionada e não-ponderada, o que a torna ideal para investigar padrões clássicos de agrupamento social, homofilia e propriedades de conectividade humana.

### Amostragem e limitações do grafo
Como os dados foram consolidados a partir de um conjunto restrito de 10 indivíduos focais (os egos), a amostra possui limitações inerentes de representatividade:
* Viés de seleção: A topologia reflete quase que inteiramente os círculos de amizade específicos das 10 pessoas escolhidas para a coleta inicial, não sendo uma amostra aleatória uniforme de toda a plataforma.
* Efeito de borda (*boundary spanning*): Conexões que os membros da rede possuem com usuários externos aos 10 círculos egocêntricos não foram coletadas. Isso tende a distorcer certas métricas de centralidade e distâncias globais na comparação com a rede real completa.
* Natureza estática: A rede representa uma fotografia instantânea e anonimizada capturada no ano de 2012, de modo que não captura a evolução temporal e novas interações dinâmicas.

### Estimativa da fração da rede coletada
Na época de compilação aproximada deste dataset pelo projeto *SNAP* (em torno de 2012), o Facebook contava com cerca de 1 bilhão de usuários ativos mensais (1.000.000.000 de nós). 
Como o subgrafo em análise possui 4.039 nós, a fração que ele representa em relação ao grafo total é estimada em:
$$\text{Fração} = \frac{4.039}{1.000.000.000} \approx 4 \times 10^{-6} = 0,0004\%$$
Isso equivale a menos de quatro milionésimos da rede global da plataforma na época. Embora a fração global seja extremamente pequena, a grande vantagem científica desse dataset egocêntrico unificado é que ele mantém uma altíssima densidade local e conexões ricas em comunidades reais fechadas, algo que seria perdido caso a amostragem fosse feita de forma esparsa ou totalmente aleatória.
